# Upscale an Image 4x with Real-ESRGAN

Downloads a small sample image from a public URL and makes it 4 times larger with Real-ESRGAN, a neural network that adds realistic detail instead of just stretching pixels. The before and after images are shown side by side and the result is saved.

## How to run
- Open in Google Colab. Runtime type: **T4 GPU**.
- Run the cells from top to bottom.

_Rewritten from my original Colab notebook: `GPT.ipynb`._

In [ ]:
!pip install -q realesrgan

## Fix a torchvision compatibility issue
`basicsr` (used by Real-ESRGAN) still imports `torchvision.transforms.functional_tensor`, which newer torchvision removed. This cell patches that one import line.

In [ ]:
import importlib.util
from pathlib import Path

# find_spec locates the package without importing it (importing would fail before the patch)
basicsr_dir = Path(importlib.util.find_spec("basicsr").origin).parent
deg_file = basicsr_dir / "data" / "degradations.py"
code = deg_file.read_text()
code = code.replace("torchvision.transforms.functional_tensor", "torchvision.transforms.functional")
deg_file.write_text(code)
print("Patched:", deg_file)

## Load the Real-ESRGAN model
The weights download automatically the first time.

In [ ]:
import torch
from basicsr.archs.rrdbnet_arch import RRDBNet
from realesrgan import RealESRGANer

# RealESRGAN_x4plus: a general-purpose 4x upscaling model
model = RRDBNet(num_in_ch=3, num_out_ch=3, num_feat=64, num_block=23, num_grow_ch=32, scale=4)
upsampler = RealESRGANer(
    scale=4,
    model_path="https://github.com/xinntao/Real-ESRGAN/releases/download/v0.1.0/RealESRGAN_x4plus.pth",
    model=model,
    tile=400,          # process the image in tiles so big images fit in GPU memory
    tile_pad=10,
    pre_pad=0,
    half=torch.cuda.is_available(),  # half precision is faster on GPU
)
print("Model ready")

In [ ]:
import numpy as np
from PIL import Image

def upscale(pil_image, outscale=4):
    """Upscale a PIL image with Real-ESRGAN and return a PIL image."""
    rgb = np.array(pil_image.convert("RGB"))
    bgr = rgb[:, :, ::-1]                      # Real-ESRGAN expects BGR (OpenCV order)
    out_bgr, _ = upsampler.enhance(bgr, outscale=outscale)
    return Image.fromarray(out_bgr[:, :, ::-1])

## Download a sample image

In [ ]:
import requests
from io import BytesIO

url = "https://upload.wikimedia.org/wikipedia/commons/thumb/3/3f/Fronalpstock_big.jpg/320px-Fronalpstock_big.jpg"
resp = requests.get(url, headers={"User-Agent": "colab-notebook"}, timeout=30)
resp.raise_for_status()
original = Image.open(BytesIO(resp.content)).convert("RGB")
print("Original size:", original.size)

## Upscale and compare

In [ ]:
import matplotlib.pyplot as plt

upscaled = upscale(original)
upscaled.save("upscaled.png")
print("Upscaled size:", upscaled.size)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
for ax, im, title in zip(axes, [original, upscaled], ["Original", "Real-ESRGAN 4x"]):
    ax.imshow(im)
    ax.set_title(f"{title} {im.size}")
    ax.axis("off")
plt.show()